# Fine-tuning LoRA Physics v2 — SD 1.5 sur Merapi (Colab GPU)

**VERSION 2 — Corrections majeures :**
- ✅ Prompts en **langage naturel riche** (CLIP comprend "bright orange lava", pas "lava_intensity=high")
- ✅ **Coloration physiquement motivée** (au lieu de gris → gris partout)
- ✅ **LoRA rank 16** (au lieu de 4 — plus de capacité conditionnelle)
- ✅ **Nouveaux paramètres** : viscosité, température, type éruption, panache
- ✅ **guidance_scale=10** pour l'inférence de test

## Étapes
1. Installer les dépendances
2. Uploader le dataset (ZIP)
3. Lancer l'entraînement avec prompts NL riches + augmentation couleur
4. Télécharger le LoRA physics v2 résultant

In [1]:
# Vérifier le GPU
!nvidia-smi

Thu Apr 16 10:52:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8              3W /   70W |      38MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Installer les dépendances
!pip install -q torch torchvision diffusers transformers accelerate peft pillow pandas

In [4]:
# 2. Uploader le dataset
# Sur votre Mac :
#   cd merapi_anomaly
#   zip -r merapi_dataset.zip data/processed/ data/index/index.csv

from google.colab import files
import zipfile
#
# print("Uploadez merapi_dataset.zip...")
# uploaded = files.upload()

with zipfile.ZipFile("merapi_dataset.zip", "r") as z:
    z.extractall("/content/merapi_anomaly/")

print("Dataset extrait.")
!find /content/merapi_anomaly/data/processed -name '*.png' | wc -l

PermissionError: [Errno 13] Permission denied: '/content'

In [10]:
import os, sys, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from PIL import Image
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

device = torch.device("cuda")
dtype = torch.float16
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.2 GB


In [ ]:
# ─── Configuration v2 ────────────────────────────────────────
RESOLUTION = 256
MAX_IMAGES = None       # None = toutes les images
EPOCHS = 50
BATCH_SIZE = 4          # 4 sur T4, 8 sur A100
LR = 5e-5              # Plus élevé pour rank 16
GRAD_ACCUM = 4
LORA_RANK = 16          # v2: 16 au lieu de 4 (plus de capacité conditionnelle)
LORA_ALPHA = 16
EMA_DECAY = 0.9999
PATIENCE = 20
SAVE_EVERY = 25
SAMPLE_EVERY = 10
CFG_DROPOUT = 0.1       # 10% du temps : prompt vide (apprend l'unconditional)
GUIDANCE_SCALE = 10.0   # v2: 10 au lieu de 7.5 (suit mieux les prompts)

DATA_ROOT = Path("./merapi_dataset/data")           # ou Path.cwd() / "merapi_anomaly/data"
OUTPUT_DIR = Path("./lora_merapi_physics_output")   # dossier créé là où vous exécutez le script
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
(OUTPUT_DIR / "samples").mkdir(exist_ok=True)

In [ ]:
# ─── Prompts NL riches + Coloration physique (v2) ────────────
# CLIP comprend le langage naturel, PAS les paires key=value.
# "bright orange-red incandescent lava" → embedding riche et distinct
# "lava_intensity=very_high" → embedding quasi-identique à "lava_intensity=none"

import random as _rnd

NEGATIVE_PROMPT = (
    "extra volcanoes, duplicate peaks, multiple mountains, fantasy landscape, "
    "buildings, houses, city, people, vehicles, text, watermark, logo, signature, "
    "cartoon, anime, drawing, painting, illustration, digital art, 3D render, "
    "low quality, blurry, pixelated, jpeg artifacts, wrong perspective, "
    "oversaturated, neon colors, unrealistic colors"
)

# ── Descriptions riches par paramètre ──
CAMERA_DESC = {
    "Suki": "south face view from Suki monitoring station, looking up at the summit cone",
    "Kalor": "west face view from Kalor station, lateral perspective showing the crater rim",
    "Kali": "east face view from Kali station, showing the eastern slope and ravines",
}
TIME_DESC = {
    "early_morning": "early morning with soft warm golden light and long shadows across the slopes",
    "midday": "bright midday under harsh overhead sunlight, strong contrast",
    "afternoon": "warm afternoon light with slightly golden tone on the volcanic terrain",
    "dusk": "dramatic dusk with orange and purple sky, the volcano silhouetted against sunset colors",
    "night": "dark nighttime",
}
BRIGHTNESS_DESC = {
    "daylight": "well-lit in natural daylight, good visibility of terrain details",
    "bright_with_incandescence": "bright daylight with visible volcanic incandescence glowing through",
    "incandescent_glow": "volcanic incandescence illuminating the dark slopes with intense orange-red glow",
    "dim_glow": "faint dim volcanic glow barely visible against the dark mountain",
    "dark": "very dark with minimal visibility, only mountain silhouette barely discernible",
}
LAVA_DESC = {
    "none": "No visible lava or volcanic activity, calm dormant slopes",
    "low": "Faint traces of lava visible as small orange-red dots scattered on the upper slope",
    "moderate": "Moderate lava flow creating visible bright orange streams down the mountainside channels",
    "high": "Intense lava flow with bright orange-yellow streams cascading down multiple drainage channels",
    "very_high": "Massive incandescent lava flow covering large portions of the slope, extremely bright white-orange hot material streaming downhill",
}
WEATHER_DESC = {
    "clear": "clear sky with excellent visibility",
    "overcast": "thick cloud cover partially obscuring the summit, diffuse soft lighting",
    "hazy": "hazy atmospheric conditions with mist and reduced visibility",
    "clear_night": "clear dark night sky",
}
VISCOSITY_DESC = {
    "low": "thin fluid rapidly flowing lava streams",
    "medium": "moderately viscous flowing lava",
    "high": "thick slow-moving viscous lava with rough blocky texture",
}
TEMPERATURE_DESC = {
    "low": "dim dark red glow indicating cooling solidifying lava",
    "moderate": "steady bright orange glow from actively flowing lava",
    "high": "bright orange-yellow incandescence from very hot volcanic material",
    "extreme": "brilliant white-hot incandescence from extremely hot freshly erupted material",
}
ERUPTION_DESC = {
    "none": "",
    "effusive": "Effusive eruption with steady continuous lava outflow from the crater.",
    "explosive": "Explosive eruption with ejected rocks and pyroclastic material flying from the summit.",
    "phreatic": "Phreatic steam eruption with dense white vapor cloud rising from the crater.",
}
PLUME_DESC = {
    "none": "",
    "low": "A small wispy steam plume rises from the crater rim.",
    "medium": "A visible gray ash plume extends above the summit into the sky.",
    "high": "A tall dense volcanic ash column rises high into the atmosphere above the mountain.",
}

_TEMPLATES = [
    ("Surveillance camera photograph of Mount Merapi volcano, {camera_desc}. "
     "{time_desc}, {weather_desc}. {brightness_desc}. {lava_desc}. "
     "{extra}Rocky volcanic terrain with gray andesitic rock. "
     "Realistic monitoring camera image, natural photographic quality."),
    ("Mount Merapi volcano monitoring webcam image, {camera_desc}. "
     "{time_desc}. {weather_desc}. {brightness_desc}. "
     "{lava_desc}. {extra}Volcanic landscape with rocky slopes. "
     "Real surveillance photograph, detailed natural image."),
    ("Webcam photograph of Merapi stratovolcano in Java, {camera_desc}. "
     "{time_desc}, {weather_desc}. {brightness_desc}. "
     "{lava_desc}. {extra}Andesitic volcanic terrain. "
     "High resolution monitoring station image."),
]

def build_rich_prompt(camera="Suki", time_of_day="midday", brightness="daylight",
                      lava_intensity="none", slope="30deg_south", weather="clear",
                      viscosity="medium", temperature="moderate",
                      eruption_type="none", plume="none", template_index=None):
    """Construit un prompt NL riche depuis les paramètres physiques."""
    camera_desc = CAMERA_DESC.get(camera, f"view from {camera} station")
    time_desc = TIME_DESC.get(time_of_day, time_of_day)
    brightness_desc = BRIGHTNESS_DESC.get(brightness, brightness)
    lava_desc = LAVA_DESC.get(lava_intensity, lava_intensity)
    weather_desc = WEATHER_DESC.get(weather, weather)
    
    extra_parts = []
    if lava_intensity not in ("none",):
        visc = VISCOSITY_DESC.get(viscosity, "")
        temp = TEMPERATURE_DESC.get(temperature, "")
        if visc and temp:
            extra_parts.append(f"{visc} with {temp}.")
    if eruption_type != "none":
        erupt = ERUPTION_DESC.get(eruption_type, "")
        if erupt:
            extra_parts.append(erupt)
    if plume != "none":
        pl = PLUME_DESC.get(plume, "")
        if pl:
            extra_parts.append(pl)
    extra = " ".join(extra_parts) + (" " if extra_parts else "")
    
    template = _TEMPLATES[template_index % len(_TEMPLATES)] if template_index is not None else _rnd.choice(_TEMPLATES)
    return template.format(
        camera_desc=camera_desc, time_desc=time_desc,
        brightness_desc=brightness_desc, lava_desc=lava_desc,
        weather_desc=weather_desc, extra=extra,
    )

# ── Helpers extraction metadata ──
def _camera(fn):
    fn = fn.lower()
    if fn.startswith("suki"): return "Suki"
    elif fn.startswith("kalor"): return "Kalor"
    elif fn.startswith("kali"): return "Kali"
    return "unknown"

def _time_of_day(hour):
    if 6 <= hour < 10: return "early_morning"
    elif 10 <= hour < 14: return "midday"
    elif 14 <= hour < 18: return "afternoon"
    elif 18 <= hour < 21: return "dusk"
    return "night"

def _brightness(hour, anomaly):
    if 6 <= hour < 18:
        return "bright_with_incandescence" if anomaly > 0.3 else "daylight"
    if anomaly > 0.3: return "incandescent_glow"
    elif anomaly > 0.1: return "dim_glow"
    return "dark"

def _lava_intensity(anomaly):
    if anomaly > 0.5: return "very_high"
    elif anomaly > 0.3: return "high"
    elif anomaly > 0.15: return "moderate"
    elif anomaly > 0.05: return "low"
    return "none"

def _slope(camera):
    return {"Suki": "30deg_south", "Kalor": "25deg_west", "Kali": "35deg_east"}.get(camera, "30deg_unknown")

def _weather(quality_flag, hour):
    if quality_flag == "cloudy": return "overcast"
    elif quality_flag == "dark": return "clear_night" if hour >= 18 or hour < 6 else "hazy"
    return "clear"

def build_prompt_from_row(row):
    """Construit un prompt NL riche depuis une ligne index.csv."""
    hour = int(row.get("hour", 12))
    anomaly = float(row.get("anomaly_score", 0.0))
    fn = str(row.get("filename", ""))
    quality = str(row.get("quality_flag", "usable"))
    cam = _camera(fn)
    
    viscosity = "low" if anomaly > 0.3 else ("medium" if anomaly > 0.1 else "high")
    temp = "high" if anomaly > 0.5 else ("moderate" if anomaly > 0.1 else "low")
    eruption = "effusive" if anomaly > 0.3 else "none"
    
    return build_rich_prompt(
        camera=cam, time_of_day=_time_of_day(hour),
        brightness=_brightness(hour, anomaly),
        lava_intensity=_lava_intensity(anomaly),
        slope=_slope(cam), weather=_weather(quality, hour),
        viscosity=viscosity, temperature=temp,
        eruption_type=eruption,
        template_index=0,  # fixe pour pré-cache
    )

# ── Coloration physiquement motivée ──
def apply_physics_color(img_gray, hour, anomaly, jitter=0.15):
    """Colore les images grises selon les conditions physiques.
    
    Au lieu de gris → gris (le modèle n'apprend aucune variation visuelle),
    on applique des teintes physiquement motivées :
    - Jour : tons chauds naturels
    - Nuit + activité : rouge/orange volcanique  
    - Nuit calme : bleu sombre
    - Crépuscule : doré/ambre
    """
    arr = np.array(img_gray, dtype=np.float32) / 255.0
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    
    if 6 <= hour < 10:       r, g, b = 1.08, 1.02, 0.88
    elif 10 <= hour < 14:    r, g, b = 1.0, 1.0, 0.95
    elif 14 <= hour < 18:    r, g, b = 1.05, 0.98, 0.88
    elif 18 <= hour < 21:    r, g, b = 1.2, 0.82, 0.6
    else:
        if anomaly > 0.3:
            glow = min(anomaly * 1.8, 1.0)
            r, g, b = 0.3 + 0.7 * glow, 0.15 + 0.35 * glow, 0.1
        elif anomaly > 0.1:
            r, g, b = 0.45, 0.3, 0.2
        else:
            r, g, b = 0.25, 0.3, 0.45
    
    rj = 1.0 + _rnd.uniform(-jitter, jitter)
    gj = 1.0 + _rnd.uniform(-jitter, jitter)
    bj = 1.0 + _rnd.uniform(-jitter, jitter)
    
    rgb = np.stack([
        np.clip(arr * r * rj, 0, 1),
        np.clip(arr * g * gj, 0, 1),
        np.clip(arr * b * bj, 0, 1),
    ], axis=-1)
    return Image.fromarray((rgb * 255).astype(np.uint8), mode="RGB")

print("✅ Prompts NL + coloration physique définis")

In [ ]:
# ─── Dataset v2 (NL prompts + coloration physique) ────────────

class MerapiPhysicsDataset(Dataset):
    def __init__(self, index_path, processed_base, resolution=256, max_images=None):
        self.resolution = resolution
        df = pd.read_csv(index_path, dtype=str, na_values=["", "None", "nan"])
        df["hour"] = pd.to_numeric(df["hour"], errors="coerce")
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df["month"] = pd.to_numeric(df["month"], errors="coerce")
        df["anomaly_score"] = pd.to_numeric(df["anomaly_score"], errors="coerce").fillna(0.0)
        df = df.dropna(subset=["hour", "year", "month", "filename"])
        df = df[df["quality_flag"] == "usable"].copy()
        
        # Exclure les nuits sans incandescence
        is_night = ~df["hour"].between(6, 17)
        has_glow = df["anomaly_score"] > 0.05
        df = df[~(is_night & ~has_glow)].copy()
        
        self.samples = []
        for _, row in df.iterrows():
            y, m, fn = row.get("year"), row.get("month"), row.get("filename", "")
            if pd.isna(y) or pd.isna(m) or not fn:
                continue
            png = processed_base / str(int(y)) / f"{int(m):02d}" / (Path(fn).stem + ".png")
            if not png.exists():
                continue
            # v2: prompt NL riche (template fixe pour pré-cache)
            prompt = build_prompt_from_row(row.to_dict())
            self.samples.append({
                "path": png, "prompt": prompt,
                "hour": int(row.get("hour", 12)),
                "anomaly_score": float(row.get("anomaly_score", 0.0)),
                "camera": _camera(fn),
            })
        
        if max_images and len(self.samples) > max_images:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(self.samples), max_images, replace=False)
            self.samples = [self.samples[i] for i in sorted(idx)]
        
        n_day = sum(1 for s in self.samples if 6 <= s["hour"] < 18)
        n_active = sum(1 for s in self.samples if s["anomaly_score"] > 0.15)
        cams = {}
        for s in self.samples:
            cams[s["camera"]] = cams.get(s["camera"], 0) + 1
        print(f"[Dataset] {len(self.samples)} images | Jour/Nuit: {n_day}/{len(self.samples)-n_day} | Actif: {n_active} | Caméras: {cams}")
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("L")
        if img.size != (self.resolution, self.resolution):
            img = img.resize((self.resolution, self.resolution), Image.LANCZOS)
        
        # ── v2 : COLORATION PHYSIQUE (au lieu de gris → gris) ──
        img_rgb = apply_physics_color(img, s["hour"], s["anomaly_score"])
        
        arr = np.array(img_rgb, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
        return {"pixel_values": tensor, "prompt": s["prompt"]}

dataset = MerapiPhysicsDataset(
    DATA_ROOT / "index" / "index.csv",
    DATA_ROOT / "processed",
    RESOLUTION, MAX_IMAGES,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
print(f"Batches/epoch: {len(dataloader)}")
print(f"\n📝 Exemple prompt NL riche :")
print(f"  {dataset.samples[0]['prompt'][:150]}...")

[Dataset] 2344 images | Jour/Nuit: 802/1542 | Actif: 1372 | Caméras: {'Suki': 1453, 'Kalor': 575, 'unknown': 316}
Batches/epoch: 586
Exemple prompt: Merapi, single lava flow, camera=Suki, time=afternoon, brightness=daylight, lava_intensity=none, slope=30deg_south, weather=clear. Realistic surveillance view. No extra volcanoes.


In [18]:
# ─── Charger SD 1.5 ──────────────────────────────────────────

tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=dtype).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

# ─── LoRA ────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0",
                    "proj_in", "proj_out", "ff.net.0.proj", "ff.net.2"],
)
unet = get_peft_model(unet, lora_config)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"LoRA: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 4578.42it/s]
CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/rhomdev/Documents/stage/test_IA/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


LoRA: 1,695,744 / 861,216,708 params (0.20%)


In [19]:
# ─── Pré-encoder les prompts ──────────────────────────────────

def encode_prompt(prompt, tokenizer, text_encoder, device):
    tokens = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt")
    with torch.no_grad():
        return text_encoder(tokens.input_ids.to(device)).last_hidden_state

prompt_cache = {}
uncond_emb = encode_prompt(UNCOND_PROMPT, tokenizer, text_encoder, device)

unique_prompts = set(s["prompt"] for s in dataset.samples)
for p in unique_prompts:
    prompt_cache[p] = encode_prompt(p, tokenizer, text_encoder, device)
print(f"{len(unique_prompts)} prompts uniques encodés")

68 prompts uniques encodés


In [20]:
# ─── EMA ──────────────────────────────────────────────────────

class EMAModel:
    def __init__(self, parameters, decay=0.9999):
        self.decay = decay
        self.shadow = {i: p.clone().detach() for i, p in enumerate(parameters)}
    
    @torch.no_grad()
    def update(self, parameters):
        for i, p in enumerate(parameters):
            self.shadow[i].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)
    
    def apply_shadow(self, parameters):
        self.backup = {i: p.clone() for i, p in enumerate(parameters)}
        for i, p in enumerate(parameters): p.data.copy_(self.shadow[i])
    
    def restore(self, parameters):
        for i, p in enumerate(parameters): p.data.copy_(self.backup[i])

ema = EMAModel([p for p in unet.parameters() if p.requires_grad], EMA_DECAY)

In [21]:
# ─── Optimiseur + Scheduler ───────────────────────────────────

params_opt = list(filter(lambda p: p.requires_grad, unet.parameters()))
optimizer = torch.optim.AdamW(params_opt, lr=LR, weight_decay=0.01)
total_steps = EPOCHS * math.ceil(len(dataloader) / GRAD_ACCUM)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=LR * 0.1)
scaler = GradScaler("cuda")
rng_cfg = np.random.default_rng(42)

In [ ]:
# ─── Scénarios de test v2 (6 scénarios très contrastés) ───────

TEST_SCENARIOS = [
    {"name": "jour_calme_clair", "params": {
        "camera": "Suki", "time_of_day": "midday", "brightness": "daylight",
        "lava_intensity": "none", "slope": "30deg_south", "weather": "clear"}},
    {"name": "nuit_eruption_intense", "params": {
        "camera": "Kalor", "time_of_day": "night", "brightness": "incandescent_glow",
        "lava_intensity": "very_high", "slope": "25deg_west", "weather": "clear_night",
        "viscosity": "low", "temperature": "extreme", "eruption_type": "effusive", "plume": "medium"}},
    {"name": "crepuscule_moderate", "params": {
        "camera": "Suki", "time_of_day": "dusk", "brightness": "dim_glow",
        "lava_intensity": "moderate", "slope": "30deg_south", "weather": "clear",
        "viscosity": "medium", "temperature": "moderate"}},
    {"name": "nuit_faible_couvert", "params": {
        "camera": "Kalor", "time_of_day": "night", "brightness": "dim_glow",
        "lava_intensity": "low", "slope": "25deg_west", "weather": "overcast",
        "viscosity": "high", "temperature": "low"}},
    {"name": "matin_brume_calme", "params": {
        "camera": "Kali", "time_of_day": "early_morning", "brightness": "daylight",
        "lava_intensity": "none", "slope": "35deg_east", "weather": "hazy"}},
    {"name": "nuit_explosive_panache", "params": {
        "camera": "Suki", "time_of_day": "night", "brightness": "incandescent_glow",
        "lava_intensity": "high", "slope": "30deg_south", "weather": "clear_night",
        "viscosity": "low", "temperature": "high", "eruption_type": "explosive", "plume": "high"}},
]

def generate_samples(unet, prefix, scenarios=TEST_SCENARIOS, seed=42):
    unet.eval()
    unet_raw = unet.base_model.model if hasattr(unet, 'base_model') else unet
    pipe = StableDiffusionPipeline(
        vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
        unet=unet_raw, scheduler=noise_scheduler,
        safety_checker=None, feature_extractor=None, requires_safety_checker=False,
    ).to(device)
    pipe.set_progress_bar_config(disable=True)
    
    gen = torch.Generator("cpu").manual_seed(seed)
    for sc in scenarios:
        # v2: prompt NL riche (au lieu de key=value)
        prompt = build_rich_prompt(**sc["params"], template_index=0)
        with torch.no_grad():
            img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
                       num_inference_steps=30, guidance_scale=GUIDANCE_SCALE,
                       generator=gen).images[0]
        path = OUTPUT_DIR / "samples" / f"{prefix}_{sc['name']}.png"
        img.save(str(path))
        print(f"  {path.name}")
    del pipe
    torch.cuda.empty_cache()
    unet.train()

print("Génération AVANT fine-tuning (6 scénarios)...")
generate_samples(unet, "before")

Génération avant fine-tuning...
  before_jour_calme_suki.png
  before_nuit_eruption_kalor.png
  before_dusk_moderate_suki.png
  before_nuit_faible_couvert.png


In [23]:
# ─── ENTRAÎNEMENT ────────────────────────────────────────────

unet.train()
best_loss = float("inf")
patience_counter = 0
global_step = 0
loss_history = []
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch_idx, batch in enumerate(dataloader):
        pixel_values = batch["pixel_values"].to(device, dtype=dtype)
        prompts = batch["prompt"]
        bsz = pixel_values.shape[0]
        
        # VAE encode
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
        
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        
        # Encoder prompts avec CFG dropout (10% → prompt vide)
        embs = []
        for p in prompts:
            if rng_cfg.random() < CFG_DROPOUT:
                embs.append(uncond_emb)
            elif p in prompt_cache:
                embs.append(prompt_cache[p])
            else:
                e = encode_prompt(p, tokenizer, text_encoder, device)
                prompt_cache[p] = e
                embs.append(e)
        encoder_hidden_states = torch.cat(embs, dim=0)
        
        # Forward + Loss (mixed precision)
        with autocast("cuda"):
            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states).sample
            loss = F.mse_loss(noise_pred.float(), noise.float()) / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params_opt, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            ema.update(p for p in unet.parameters() if p.requires_grad)
            global_step += 1
        
        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches += 1
    
    avg_loss = epoch_loss / max(n_batches, 1)
    loss_history.append(avg_loss)
    lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t_start
    print(f"Epoch {epoch:03d}/{EPOCHS} | Loss={avg_loss:.6f} | LR={lr:.2e} | Step={global_step} | {elapsed/60:.1f}min")
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        unet.save_pretrained(str(OUTPUT_DIR / "checkpoints" / "best_lora"))
        print(f"  ★ Best (loss={best_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping ({PATIENCE} epochs sans amélioration).")
            break
    
    if epoch % SAVE_EVERY == 0:
        unet.save_pretrained(str(OUTPUT_DIR / "checkpoints" / f"lora_epoch_{epoch:03d}"))
    
    if epoch % SAMPLE_EVERY == 0:
        ema.apply_shadow(p for p in unet.parameters() if p.requires_grad)
        generate_samples(unet, f"e{epoch:03d}", TEST_SCENARIOS[:2])
        ema.restore(p for p in unet.parameters() if p.requires_grad)
        unet.train()

print(f"\nTerminé en {(time.time()-t_start)/60:.1f} min | Best loss={best_loss:.6f}")

Epoch 001/100 | Loss=0.132988 | LR=5.00e-06 | Step=146 | 2.2min
  ★ Best (loss=0.132988)
Epoch 002/100 | Loss=0.118034 | LR=5.00e-06 | Step=292 | 4.5min
  ★ Best (loss=0.118034)
Epoch 003/100 | Loss=0.123596 | LR=4.99e-06 | Step=438 | 6.7min
Epoch 004/100 | Loss=0.120104 | LR=4.98e-06 | Step=584 | 8.9min
Epoch 005/100 | Loss=0.121113 | LR=4.97e-06 | Step=730 | 11.2min
Epoch 006/100 | Loss=0.117374 | LR=4.96e-06 | Step=876 | 13.4min
  ★ Best (loss=0.117374)
Epoch 007/100 | Loss=0.123025 | LR=4.95e-06 | Step=1022 | 15.6min
Epoch 008/100 | Loss=0.115555 | LR=4.93e-06 | Step=1168 | 17.9min
  ★ Best (loss=0.115555)
Epoch 009/100 | Loss=0.119875 | LR=4.91e-06 | Step=1314 | 20.1min
Epoch 010/100 | Loss=0.121767 | LR=4.89e-06 | Step=1460 | 22.3min
  e010_jour_calme_suki.png
  e010_nuit_eruption_kalor.png
Epoch 011/100 | Loss=0.115771 | LR=4.87e-06 | Step=1606 | 24.7min
Epoch 012/100 | Loss=0.116028 | LR=4.84e-06 | Step=1752 | 26.9min
Epoch 013/100 | Loss=0.118315 | LR=4.82e-06 | Step=1898 | 29

In [24]:
# ─── Sauvegarde finale (EMA) ──────────────────────────────────

ema.apply_shadow(p for p in unet.parameters() if p.requires_grad)
final_path = OUTPUT_DIR / "lora_merapi_physics_final"
unet.save_pretrained(str(final_path))
print(f"LoRA final → {final_path}")

# Loss CSV
pd.DataFrame({"epoch": range(1, len(loss_history)+1), "loss": loss_history}).to_csv(
    OUTPUT_DIR / "training_loss.csv", index=False)

# Derniers échantillons
print("\nGénération finale...")
generate_samples(unet, "final")

ema.restore(p for p in unet.parameters() if p.requires_grad)

LoRA final → lora_merapi_physics_output/lora_merapi_physics_final

Génération finale...
  final_jour_calme_suki.png
  final_nuit_eruption_kalor.png
  final_dusk_moderate_suki.png
  final_nuit_faible_couvert.png


In [28]:
# ─── Télécharger les résultats ────────────────────────────────

import shutil
shutil.make_archive("outputs/lora_merapi_physics_results", "zip", str(OUTPUT_DIR))
# files.download("/lora_merapi_physics_results.zip")
print("Téléchargement lancé !")
print("\nPlacez le contenu dans : merapi_anomaly/outputs/lora_merapi_physics_results/")

Téléchargement lancé !

Placez le contenu dans : merapi_anomaly/outputs/lora_merapi_physics_results/
